# Дневная оценка и похожие утра — CPU

Диагностический запуск без переобучения баз и без сабмита. Изолируем поправку объёма Session6: доли Session6 одинаковы во всех вариантах. Настройки выбираются на 4–17 октября, test 18–31 октября оценивается после фиксации. Дополнительно показываем перенос выбранной поправки на архивные доли бленда. Это дополнительная проверка, а не основание повторно подбирать параметры.

Архивные базы не воспроизводят новый full-history refit. Перед конкурсным сабмитом нужен отдельный полный refit выбранного решения. Этот ноутбук проверяет гипотезу и сохраняет результаты, не изменяя рабочие сабмиты.

В первой ячейке при необходимости укажите пути. На сервере ищутся файлы в распакованном комплекте session6_blend_server. Нужны base_oof.parquet, test_predictions.parquet исходного бленда и train_solo_track.parquet. При нескольких разных копиях укажите пути явно.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path(os.environ.get('BLEND_PACKAGE', '/root/experiments/session6_blend_server'))
OOF_PATH = None
TEST_PATH = None
HISTORY_PATH = None
OUT = ROOT / 'output_transfer_cpu'
OUT.mkdir(parents=True, exist_ok=True)
def locate(explicit, name, required):
    if explicit is not None:
        return Path(explicit)
    import pyarrow.parquet as pq
    candidates = []
    for p in ROOT.rglob(name):
        if OUT in p.parents:
            continue
        if set(required).issubset(pq.ParquetFile(p).schema.names):
            candidates.append(p)
    if len(candidates) != 1:
        raise ValueError(f'Укажите путь явно: {name}: {candidates}')
    return candidates[0]
OOF_PATH = locate(OOF_PATH, 'base_oof.parquet', ['session6', 'y', 'cutoff'])
TEST_PATH = locate(TEST_PATH, 'test_predictions.parquet', ['cat_d4__fixed', 'session6', 'y'])
HISTORY_PATH = locate(HISTORY_PATH, 'train_solo_track.parquet', ['target_1h', 'status_5'])
print(OOF_PATH, TEST_PATH, HISTORY_PATH, sep='\n')
oof = pd.read_parquet(OOF_PATH)
test = pd.read_parquet(TEST_PATH)
for f in [oof, test]:
    f['cutoff'] = pd.to_datetime(f.cutoff)
    if f.duplicated(['cutoff','horizon','route_id']).any():
        raise ValueError('Повторяющиеся ключи')
if not oof.cutoff.lt('2025-10-18').all() or not test.cutoff.between('2025-10-18','2025-10-31 23:59:59').all():
    raise ValueError('Неверные временные границы')


In [ ]:
columns = ['target_1h'] + [f'status_{i}' for i in range(1,7)]
history = pd.read_parquet(HISTORY_PATH, columns=['timestamp']+columns)
network = history.groupby('timestamp')[columns].sum().astype(float)
network.index = pd.to_datetime(network.index)
m = network.loc[(network.index.hour>=6)&((network.index.hour<10)|((network.index.hour==10)&(network.index.minute<=30)))].copy()
m['day'] = m.index.normalize()
m['slot'] = m.index.hour*2+m.index.minute//30
features = np.log1p(m.pivot(index='day',columns='slot',values=columns))
if features.isna().any().any():
    raise ValueError('Неполное утро')
def aggregate(f):
    a = f.groupby(['cutoff','horizon'])[['y','session6']].sum().astype(float).reset_index()
    a['day'] = a.cutoff.dt.normalize()
    a['error'] = a.y/a.session6-1
    if a.session6.le(0).any():
        raise ValueError('Нулевой объём')
    return a
memory = aggregate(oof)
def correction(day, library, k, strength, cap):
    legal = library.loc[library.cutoff + pd.to_timedelta(library.horizon*30,unit='m') < day]
    days = pd.DatetimeIndex(legal.day.unique())
    if strength == 0 or len(days)<k:
        return np.zeros(8)
    x = features.loc[days]
    scale = x.std().clip(lower=.05)
    distance = (((x-features.loc[day])/scale)**2).mean(axis=1).pow(.5)
    neighbours = distance.nsmallest(k).index
    errors = legal.loc[legal.day.isin(neighbours)].pivot(index='day',columns='horizon',values='error').reindex(columns=range(1,9))
    if errors.isna().any().any():
        raise ValueError('Неполные горизонты соседей')
    median = errors.median().to_numpy()
    agreement = np.maximum((errors>0).mean(),(errors<0).mean()).to_numpy()
    return np.where(agreement>=.7,np.clip(strength*median,-cap,cap),0)
def metrics(y,p):
    e=np.asarray(p,dtype=float)-np.asarray(y,dtype=float)
    total=np.asarray(y,dtype=float).sum()
    return dict(WAPE=float(np.abs(e).sum()/total),bias=float(e.sum()/total),score=float((np.abs(e).sum()+abs(e.sum()))/total))
def evaluate(frame, library, config, shape='session6'):
    f=frame.copy()
    total=f.groupby(['cutoff','horizon'])[shape].transform('sum')
    volume=f.groupby(['cutoff','horizon']).session6.transform('sum')
    changes={d:correction(d,library,*config) for d in f.cutoff.dt.normalize().unique()}
    factors=np.array([1+changes[d][int(h)-1] for d,h in zip(f.cutoff.dt.normalize(),f.horizon)])
    pred=f[shape].to_numpy()/total.to_numpy()*volume.to_numpy()*factors
    daily=pd.DataFrame([dict(day=str(d.date()),**metrics(g.y,pred[g.index])) for d,g in f.reset_index(drop=True).groupby(f.cutoff.dt.normalize().to_numpy())])
    summary=dict(**metrics(f.y,pred),daily_mean=float(daily.score.mean()),daily_p90=float(daily.score.quantile(.9)))
    return summary,daily,pred


In [ ]:
configs=[(5,0.,.005)]+[(k,s,c) for k in [5,10] for s in [.25,.5] for c in [.005,.01]]
rows=[]
for boundary,end in [('2025-10-04','2025-10-11'),('2025-10-11','2025-10-18')]:
    library=memory.loc[memory.cutoff<pd.Timestamp(boundary)]
    val=oof.loc[(oof.cutoff>=boundary)&(oof.cutoff<end)].reset_index(drop=True)
    for config in configs:
        stats,_,_=evaluate(val,library,config)
        rows.append(dict(week=boundary,config=json.dumps(config),**stats))
validation=pd.DataFrame(rows)
ranking=validation.groupby('config')[['score','daily_mean','daily_p90']].mean().sort_values(['daily_mean','daily_p90'])
winner=tuple(json.loads(ranking.index[0]))
(OUT/'frozen_selection.json').write_text(json.dumps(dict(config=winner,criterion='mean daily score over two pretest folds',test_used_for_selection=False),indent=2))
validation.to_csv(OUT/'validation.csv',index=False)
display(validation)
display(ranking)
print('Замороженный выбор:',winner)


In [ ]:
rows=[]
predictions=test[['cutoff','route_id','horizon','y']].copy()
for label,config in [('control',configs[0]),('selected',winner)]:
    for period,frame in [('train_prequential',oof.reset_index(drop=True)),('test',test.reset_index(drop=True))]:
        stats,daily,pred=evaluate(frame,memory,config)
        rows.append(dict(period=period,variant=label,**stats))
        daily.to_csv(OUT/f'{period}_{label}_daily.csv',index=False)
        if period=='test':
            predictions[label]=pred
    stats,daily,pred=evaluate(test.reset_index(drop=True),memory,config,'cat_d4__fixed')
    rows.append(dict(period='test_archived_blend_shape',variant=label,**stats))
    predictions[label+'_blend_shape']=pred
results=pd.DataFrame(rows)
results.to_csv(OUT/'metrics.csv',index=False)
predictions.to_parquet(OUT/'test_predictions.parquet',index=False)
display(results)
print('Оценка test до refit. Этот test уже изучался; новой независимой оценкой результат не является.')
print('train_prequential: только более ранние дни в памяти; первые дни используют нулевую поправку.')
print('Результаты:',OUT)
